# ⚡ Vectorisation — why bother with numpy at all

**Time:** ~35 minutes &nbsp;•&nbsp; **Goal:** understand the one idea that makes numpy worth learning — and measure it yourself.

You could have done almost everything in the last three notebooks with lists and loops. It would have been longer to write, but it would have worked.

So why does every scientist in the building use numpy? Two reasons, and this notebook demonstrates both:

1. **It is shorter.** You write the operation you mean, not the machinery to apply it.
2. **It is dramatically faster.** Not 10% faster. Tens of times faster.

The name for "do this to the whole array at once, without a loop" is **vectorisation**.

## Our data — one neuron, and a big pile of numbers

Run this first.

- `dff`, `go`, `trials` — as before
- `big` — two million random numbers, purely so you have something large enough to time

> Two million sounds like a lot. One afternoon of imaging a few hundred neurons produces far more.

In [ ]:
# Run this first.
import time
import urllib.request
from pathlib import Path

import numpy as np

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab05/data/"

NEURON = "neuron_0588_20180828_5.csv"
if not Path(NEURON).exists():
    urllib.request.urlretrieve(DATA_URL + NEURON, NEURON)

dff, cue, go, nogo, reward = np.loadtxt(
    NEURON, delimiter=",", skiprows=1, usecols=(1, 2, 3, 4, 5), unpack=True
)

FRAME_RATE = 10.8
BEFORE, AFTER = 10, 30


def onsets_of(marker):
    return np.where(np.diff(marker) == 1)[0] + 1


def windows_of(marker):
    starts = onsets_of(marker)
    return [dff[s - BEFORE:s + AFTER] for s in starts
            if s >= BEFORE and s + AFTER <= len(dff)]


trials = np.stack(windows_of(go))       # (254, 40)

# a deliberately large array, for the timing section
big = np.random.default_rng(0).standard_normal(2_000_000)

print("trials:", trials.shape)
print("big:   ", big.shape)

## Skill 1 — The same job, written two ways

Say you want to double every value. With a list you must describe the *machinery*: make somewhere to put the answers, walk the collection, handle each element, put it back.

```python
doubled = []
for value in values:
    doubled.append(value * 2)
```

With an array you describe the *operation*:

```python
doubled = values * 2
```

Both give the same numbers. The second says what you actually meant, and it is the version you can still read in six months.

In [ ]:
# DEMO - same answer, two ways
values = dff[:10]

# the loop way
doubled_loop = []
for v in values:
    doubled_loop.append(v * 2)

# the vectorised way
doubled_array = values * 2

print("loop: ", np.round(doubled_loop, 3))
print("array:", doubled_array.round(3))
print("identical?", np.allclose(doubled_loop, doubled_array))

### ⚡ DIY challenge

Rewrite each of these loops as a single line of array arithmetic, and check with `np.allclose`:

1. `[v + 1 for v in dff]`
2. `[v ** 2 for v in dff]`
3. `[a - b for a, b in zip(dff[1:], dff[:-1])]` — the change from one frame to the next.

> 💡 Number 3 has a built-in that does it for you. You met it in notebook 2.

In [ ]:
# TODO - your turn

## Skill 2 — Measure it

Claims about speed are worth nothing without a measurement. `time.perf_counter()` returns a number of seconds; call it before and after and subtract.

```python
start = time.perf_counter()
...
elapsed = time.perf_counter() - start
```

Run the demo below. **Your numbers will differ from mine** — that is the point. Read the speedup, not the absolute times.

In [ ]:
# DEMO - two million numbers, doubled and summed
values_list = big.tolist()          # the same numbers, as a plain Python list

start = time.perf_counter()
total = 0.0
for v in values_list:
    total += v * 2
loop_time = time.perf_counter() - start

start = time.perf_counter()
total_np = (big * 2).sum()
numpy_time = time.perf_counter() - start

print(f"loop:  {loop_time:.4f} s")
print(f"numpy: {numpy_time:.4f} s")
print(f"speedup: {loop_time / numpy_time:.0f}x")
print("same answer?", np.isclose(total, total_np))

### ⚡ DIY challenge

1. Run the demo cell a second time. Do you get exactly the same numbers? Why not?
2. Compare memory: `big.nbytes` against `sys.getsizeof(values_list)`. (`import sys` first.) The list is several times larger — and that is *before* counting the numbers it points at.
3. Cut `big` to 200,000 values and time it again. Does the **speedup** stay roughly the same?

> 💡 The list stores 2 million pointers to 2 million separate Python float objects, scattered across memory. The array stores 2 million numbers, side by side. That layout is the whole story.

In [ ]:
# TODO - your turn

## Skill 3 — Broadcasting: arrays of different shapes

Vectorisation is not only array-with-number. Two arrays combine element by element:

```python
a + b        # element 0 with element 0, element 1 with element 1, ...
```

And when the shapes don't match, numpy **stretches** the smaller one to fit — if it can. That is called **broadcasting**:

```python
trials.shape          # (254, 40)
trials - trials.mean(axis=0)     # (40,) is stretched across all 254 rows
```

The rule: numpy lines the shapes up from the **right**, and a dimension of size 1 (or missing) is repeated. `(254, 40)` and `(40,)` are compatible. `(254, 40)` and `(254,)` are **not** — and that error message will find you sooner or later.

In [ ]:
# DEMO - subtract the average trial from every trial
average_trial = trials.mean(axis=0)          # (40,)

print("trials:       ", trials.shape)
print("average_trial:", average_trial.shape)

residuals = trials - average_trial           # (254, 40) - (40,) -> broadcast
print("residuals:    ", residuals.shape)
print("mean residual:", round(residuals.mean(), 12), "<- essentially zero, as it must be")

### ⚡ DIY challenge

1. Compute `per_trial = trials.mean(axis=1)` and print its shape — one number per trial.
2. Now try `trials - per_trial`. Read the error. Which rule did it break?
3. Fix it with `per_trial[:, np.newaxis]`, which turns `(254,)` into `(254, 1)`. Print the shape and confirm it works.

> 💡 `(254, 40)` and `(254,)` fail because numpy aligns from the right: 40 against 254. Reshaping to `(254, 1)` puts the 254 where it belongs.

In [ ]:
# TODO - your turn

## Skill 4 — A real analysis, without a single loop 🏆

Here is the payoff. Two standard steps in any trial-based analysis:

- **Baseline correction** — every trial starts at a slightly different level, so subtract each trial's *own* pre-cue average. Now zero means "no change from baseline".
- **Z-scoring** — express activity in standard deviations, so neurons recorded at different brightnesses can be compared.

Both are one line each.

In [ ]:
# DEMO - baseline-correct all 254 trials at once
baseline = trials[:, :BEFORE].mean(axis=1)          # (254,) one per trial
corrected = trials - baseline[:, np.newaxis]        # broadcast down the rows

print("baseline per trial:", baseline.shape)
print("corrected:         ", corrected.shape)
print("mean of the corrected baseline:", round(corrected[:, :BEFORE].mean(), 12))
print()

average = corrected.mean(axis=0)
peak_at = average[BEFORE:].argmax() / FRAME_RATE
print(f"peak response {average[BEFORE:].max():+.3f} at {peak_at:.1f}s after the cue")

### 🏆 DIY finale — compare two neurons properly

1. Write a function `baseline_correct(trial_table)` that returns the baseline-corrected table. Two lines.
2. Write `z_score(trace)` returning `(trace - trace.mean()) / trace.std()`. One line.
3. Apply both to the `go` trials and the `reward` trials, and print the peak of each.
4. Now load a **different** neuron — `neuron_0406_20180620_8.csv` — and run the same analysis on its go trials.

Compare the two. One of these neurons is choosy about what it responds to. The other responds to everything you show it.

> 💡 Change `NEURON` and re-run the loader cell. Everything downstream is written in terms of `dff`, so it just works — which is itself a small lesson about writing analysis code.

In [ ]:
# TODO - your turn

## 🔭 Explore further

- Broadcasting, with pictures: <https://numpy.org/doc/stable/user/basics.broadcasting.html>
- Why numpy is fast — contiguous memory and compiled inner loops: <https://numpy.org/doc/stable/user/whatisnumpy.html>
- In Jupyter, `%timeit some_expression` does careful repeated timing for you — better than `perf_counter` for short operations.
- [ link — Helfrid to add a vectorisation write-up ]

## Recap

You can now:

- replace a loop with a single array expression, and say why that is clearer
- **measure** the difference rather than assert it
- explain where the speed comes from: one type, one contiguous block, one compiled loop
- use **broadcasting** to combine arrays of different shapes, and fix the shape when it fails
- baseline-correct and z-score a whole table of trials without writing a loop

**That is Lab 5's notebooks done.** In the tool session you leave Colab, set up a real Python project with `uv`, install numpy yourself, and run this analysis over **all five neurons** on your own machine.